# Agentic-SDK 實作展示

本 Notebook 將引導您了解 **Agentic-SDK** 的核心功能。這是一個以 Workflow 為核心的 Python 框架，旨在讓開發者能輕鬆組裝 Agent 的感知 (Perceive)、檢索 (Retrieve) 與行動 (Action) 行為。

## 1. 環境建置

首先，我們從 GitHub 複製專案原始碼並切換到專案目錄中。

In [7]:
!git clone https://github.com/R300-AI/Agentic-SDK.git
%cd Agentic-SDK

Cloning into 'Agentic-SDK'...
remote: Enumerating objects: 2389, done.
remote: Counting objects: 100% (831/831), done.
remote: Compressing objects: 100% (569/569), done.
remote: Total 2389 (delta 354), reused 673 (delta 227), pack-reused 1558 (from 1)
Receiving objects: 100% (2389/2389), 7.96 MiB | 29.64 MiB/s, done.
Resolving deltas: 100% (1277/1277), done.
/content/Agentic-SDK/Agentic-SDK


### 安裝依賴套件

接下來，安裝 SDK 運作所需的 Python 套件。這些套件包含 OpenAI、FastAPI 以及 Azure 相關 SDK。

In [8]:
!python -m pip install -r requirements.txt

## 2. 驗證 SDK 安裝

嘗試導入 `agentic_sdk` 以確認安裝是否成功。

In [9]:
import agentic_sdk
print('Agentic SDK import ok')

Agentic SDK import ok


## 3. Workflow 基本架構範例

這個範例展示了 SDK 的核心邏輯：
1. **Perceive**: 接收使用者輸入。
2. **Retrieve**: 透過關鍵字檢索預設的知識庫內容。
3. **Action**: 直接輸出檢索到的結果。

我們將測試一個關於「TSiP」定義的問題。

In [10]:
# 公開介面範例
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

workflow = Workflow(
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(
        items=[
            {
                "keywords": ["agentic sdk", "sdk"],
                "content": "Agentic SDK 是一個以 workflow 組裝 agent 行為的 Python library。",
            },
            {
                "keywords": ["tsip"],
                "content": "TSiP 是工研院主導的國產 AI 晶片落地藍圖。",
            },
        ],
    ),
    action=DirectAnswerAction(),
)

result = workflow.run("TSiP 是什麼？")
print(result.final_message)

TSiP 是工研院主導的國產 AI 晶片落地藍圖。


## 4. LLM 模型連線測試

在進入更複雜的 Agent 行為前，我們先直接呼叫 OpenAI 相容的 API 端點，確認大語言模型 (LLM) 能正常回應。

In [11]:
import os
from openai import OpenAI

endpoint = "https://agentic-sdk-foundry.cognitiveservices.azure.com/openai/v1/"
deployment = "agentic-sdk-gpt-5.4"
api_key = "BgNwTLZ2V2YqIT1WhPuC0CG50xcwawFyWEX5FLIRR7OQE41KT6dKJQQJ99CFACqBBLyXJ3w3AAAAACOGftJg"

client = OpenAI(
    base_url=endpoint,
    api_key= api_key,
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "I am going to Paris, what should I see?"},
    ],
    max_completion_tokens=16384,
)

print(response.choices[0].message.content)

If it’s your first time in Paris, a good mix is:

- Eiffel Tower — classic, especially at night
- Louvre Museum — huge, iconic; book ahead
- Musée d’Orsay — great for Impressionists
- Notre-Dame area — even if just to walk around Île de la Cité
- Sainte-Chapelle — stunning stained glass
- Montmartre & Sacré-Cœur — great views, charming streets
- Seine river walk or boat cruise — very Parisian
- Le Marais — cafés, boutiques, beautiful streets
- Luxembourg Gardens — relaxing and elegant
- Arc de Triomphe & Champs-Élysées — touristy but iconic

If you want a few less-obvious spots:
- Canal Saint-Martin
- Père Lachaise Cemetery
- Rue Crémieux
- Palais-Royal
- Musée Rodin

Good food experiences:
- Try a neighborhood café breakfast
- Have pastries from a good boulangerie
- Visit a street market
- Book one nice dinner, but keep some meals casual

If you tell me:
1. how many days you have,
2. whether you like art/history/food/nightlife,
3. and your budget,
I can make you a personalized Paris i

## 5. 進階應用：結合 LLM 的生成式行動 (Generative Action)

最後，我們將 Workflow 中的 `Action` 模組替換為 `GenerativeAction`。這會讓 Agent 不只是複製檢索到的內容，而是能根據 `system_prompt` 的指示，以更自然、更具結構化的方式回答使用者的問題。

In [12]:
from agentic_sdk import Workflow
from agentic_sdk.modules import GenerativeAction, KeywordRetrieve, PassThroughPerceive

workflow = Workflow(
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(
        items=[
            {
                "keywords": ["tsip"],
                "content": "TSiP 是工研院主導的國產 AI 晶片落地藍圖。",
            },
        ],
    ),
    action=GenerativeAction(
        base_url=endpoint,
        api_key=api_key,
        model=deployment,
        system_prompt=(
            "你是 Agentic SDK demo 的 Action 模組。請根據 retrieved_context 用自然語氣回答，"
            "不要逐字照抄，也不要加入 retrieved_context 沒有的外部事實。"
            "請輸出兩句繁體中文：第一句回答問題，第二句用『簡單說，』開頭做一句補充說明。"
        )
    ),
)

result = workflow.run("TSiP 是什麼？")
print(result.final_message)

TSiP 是由工研院主導推動的國產 AI 晶片落地藍圖。  
簡單說，它是在規劃並推進國產 AI 晶片實際落地應用的方向。
